In [ ]:
from google.colab import drive
import os, sys, shutil
from pathlib import Path

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Project Paths
PROJECT_NAME = "pothole_project"
DRIVE_ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
SCRIPTS_DIR = DRIVE_ROOT / "notebooks"
DRIVE_PROCESSED = DRIVE_ROOT / "processed_dataset"

# 3. Local Workspace (Ephemeral)
LOCAL_RAW = Path("/content/raw_data")
LOCAL_WORK = Path("/content/workspace")
LOCAL_OUT = Path("/content/yolo_dataset")

for p in [LOCAL_RAW, LOCAL_WORK, LOCAL_OUT]:
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True)

# 4. Add Scripts to Path (Crucial for importing your utils)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print("✅ Environment paths and sys.path configured.")

In [ ]:
!pip install -q ultralytics imagehash python-dotenv kaggle opencv-python
print("✅ Dependencies installed.")

In [ ]:

os.environ['KAGGLE_USERNAME'] = "your_username" 
os.environ['KAGGLE_KEY'] = "your_api_key"

import utils.downloader
DATASET_SLUG = "aliabdelmenam/rdd-2022"

print(f"⏬ Downloading {DATASET_SLUG} using existing downloader...")
success = utils.downloader.download_dataset(DATASET_SLUG, LOCAL_RAW)

if success:
    print("✅ Download and extraction complete.")
    print("\n📁 Extracted Structure:")
    !ls -R {str(LOCAL_RAW)} | head -n 20
else:
    print("❌ Download failed. Check your credentials and DATASET_SLUG.")

In [ ]:
os.environ['RDD2022_ROOT'] = str(LOCAL_RAW / "rdd2022") 
os.environ['WORK_DIR'] = str(LOCAL_WORK)
os.environ['YOLO_DATASET_ROOT'] = str(LOCAL_OUT)
os.environ['LOG_DIR'] = str(LOCAL_WORK / "logs")

print("🚀 Starting Preprocessing Pipeline...")
try:
    import preprocess_rdd2022 
    preprocess_rdd2022.main()
    print("\n✅ Pipeline completed successfully.")
except Exception as e:
    print(f"❌ Pipeline Error: {e}")

In [ ]:

yaml_str = f"""
path: {str(DRIVE_PROCESSED)}
train: images/train
val: images/val
test: images/test

names:
  0: Pothole
"""
with open(LOCAL_OUT / "data.yaml", "w") as f:
    f.write(yaml_str.strip())

print(f"📤 Syncing to {DRIVE_PROCESSED}...")
!rsync -ah --progress {str(LOCAL_OUT)}/ {str(DRIVE_PROCESSED)}/

print("\n✨ All done! Dataset ready at:", DRIVE_PROCESSED)